In [27]:
import pandas as pd
import torch
import os
import json
import pickle
from collections import Counter
from typing import List, Tuple, Dict, Optional



In [28]:
# Configuration
MAX_SEQ_LEN = 50
MIN_WORD_FREQ = 2
MAX_VOCAB_SIZE = 15000

# Special tokens
PAD_TOKEN = '<pad>'
SOS_TOKEN = '<sos>'
EOS_TOKEN = '<eos>'
UNK_TOKEN = '<unk>'
SPECIAL_TOKENS = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN]

print("✅ Imports loaded")
print(f"Special tokens: {SPECIAL_TOKENS}")

✅ Imports loaded
Special tokens: ['<pad>', '<sos>', '<eos>', '<unk>']


In [29]:
class TranslationTokenizer:
    def __init__(self, max_len: int = MAX_SEQ_LEN):
        self.max_len = max_len
        self.PAD_TOKEN = PAD_TOKEN
        self.SOS_TOKEN = SOS_TOKEN
        self.EOS_TOKEN = EOS_TOKEN
        self.UNK_TOKEN = UNK_TOKEN
        self.special_tokens = SPECIAL_TOKENS
        
        self.ar_stoi = None
        self.ar_itos = None
        self.en_stoi = None
        self.en_itos = None
    
    def get_special_id(self, token_type: str) -> int:
        """Get ID for special token (pad, sos, eos, unk)"""
        mapping = {
            'pad': self.PAD_TOKEN,
            'sos': self.SOS_TOKEN,
            'eos': self.EOS_TOKEN,
            'unk': self.UNK_TOKEN
        }
        return self.ar_stoi[mapping[token_type]]
    
    def tokenize(self, sentence: str, lang: str) -> List[str]:
        """Simple whitespace tokenization"""
        return sentence.split()
    
    def build_vocabulary(self, arabic_sentences: List[str], english_sentences: List[str], 
                         min_freq: int = MIN_WORD_FREQ, max_size: int = MAX_VOCAB_SIZE):
        """Build vocabularies from training data"""
        
        # Count frequencies
        ar_counter = Counter()
        for sent in arabic_sentences:
            ar_counter.update(sent.split())
        
        en_counter = Counter()
        for sent in english_sentences:
            en_counter.update(sent.split())
        
        # Build vocab with special tokens
        ar_vocab = self.special_tokens.copy()
        en_vocab = self.special_tokens.copy()
        
        # Add Arabic words by frequency
        for word, freq in ar_counter.most_common(max_size):
            if freq >= min_freq:
                ar_vocab.append(word)
        
        # Add English words by frequency
        for word, freq in en_counter.most_common(max_size):
            if freq >= min_freq:
                en_vocab.append(word)
        
        # Create mappings
        self.ar_stoi = {word: idx for idx, word in enumerate(ar_vocab)}
        self.ar_itos = ar_vocab
        self.en_stoi = {word: idx for idx, word in enumerate(en_vocab)}
        self.en_itos = en_vocab
        
        print(f"✅ Arabic vocab size: {len(self.ar_stoi)}")
        print(f"✅ English vocab size: {len(self.en_stoi)}")
    
    def numericalize(self, tokens: List[str], lang: str) -> List[int]:
        """Convert tokens to indices"""
        stoi = self.ar_stoi if lang == 'ar' else self.en_stoi
        unk_id = stoi[self.UNK_TOKEN]
        return [stoi.get(token, unk_id) for token in tokens]
    
    def add_special_tokens(self, indices: List[int], add_sos: bool = True, 
                          add_eos: bool = True) -> List[int]:
        """Add SOS and EOS tokens"""
        result = []
        if add_sos:
            result.append(self.get_special_id('sos'))
        result.extend(indices)
        if add_eos:
            result.append(self.get_special_id('eos'))
        return result
    
    def pad_sequence(self, sequence: List[int]) -> List[int]:
        """Pad or truncate sequence to max_len"""
        pad_id = self.get_special_id('pad')
        
        # Truncate if too long
        if len(sequence) > self.max_len:
            # Keep SOS and EOS
            if sequence[0] == self.get_special_id('sos'):
                sequence = sequence[:self.max_len-1] + [sequence[-1]]
            else:
                sequence = sequence[:self.max_len]
        
        # Pad if too short
        if len(sequence) < self.max_len:
            sequence = sequence + [pad_id] * (self.max_len - len(sequence))
        
        return sequence
    
    def encode_sentence(self, sentence: str, lang: str, add_sos: bool = True, 
                       add_eos: bool = True, pad: bool = True) -> torch.Tensor:
        """Full encoding pipeline"""
        tokens = self.tokenize(sentence, lang)
        indices = self.numericalize(tokens, lang)
        if add_sos or add_eos:
            indices = self.add_special_tokens(indices, add_sos, add_eos)
        if pad:
            indices = self.pad_sequence(indices)
        return torch.tensor(indices, dtype=torch.long)
    
    def decode(self, indices: List[int], lang: str, skip_special: bool = True) -> str:
        """Convert indices back to sentence"""
        itos = self.ar_itos if lang == 'ar' else self.en_itos
        tokens = []
        for idx in indices:
            token = itos[idx]
            if skip_special and token in self.special_tokens:
                continue
            tokens.append(token)
        return ' '.join(tokens)
    
    def process_dataframe(self, df: pd.DataFrame, lang: str) -> torch.Tensor:
        """Process entire dataframe column"""
        column = 'arabic' if lang == 'ar' else 'english'
        sentences = df[column].tolist()
        
        encoded = []
        for sent in sentences:
            encoded_tensor = self.encode_sentence(sent, lang)
            encoded.append(encoded_tensor)
        
        return torch.stack(encoded)
    
    def save(self, save_dir: str):
        """Save tokenizer to disk"""
        os.makedirs(save_dir, exist_ok=True)
        
        tokenizer_state = {
            'max_len': self.max_len,
            'ar_stoi': self.ar_stoi,
            'ar_itos': self.ar_itos,
            'en_stoi': self.en_stoi,
            'en_itos': self.en_itos,
            'special_tokens': self.special_tokens
        }
        
        with open(os.path.join(save_dir, 'tokenizer.pkl'), 'wb') as f:
            pickle.dump(tokenizer_state, f)
        
        # Save as JSON for inspection
        with open(os.path.join(save_dir, 'ar_vocab.json'), 'w', encoding='utf-8') as f:
            json.dump({'stoi': self.ar_stoi, 'itos': self.ar_itos}, 
                     f, ensure_ascii=False, indent=2)
        
        with open(os.path.join(save_dir, 'en_vocab.json'), 'w', encoding='utf-8') as f:
            json.dump({'stoi': self.en_stoi, 'itos': self.en_itos}, 
                     f, ensure_ascii=False, indent=2)
        
        print(f"✅ Tokenizer saved to {save_dir}")
    
    @classmethod
    def load(cls, load_dir: str):
        """Load tokenizer from disk"""
        with open(os.path.join(load_dir, 'tokenizer.pkl'), 'rb') as f:
            tokenizer_state = pickle.load(f)
        
        tokenizer = cls(max_len=tokenizer_state['max_len'])
        tokenizer.ar_stoi = tokenizer_state['ar_stoi']
        tokenizer.ar_itos = tokenizer_state['ar_itos']
        tokenizer.en_stoi = tokenizer_state['en_stoi']
        tokenizer.en_itos = tokenizer_state['en_itos']
        tokenizer.special_tokens = tokenizer_state['special_tokens']
        
        print(f"✅ Tokenizer loaded from {load_dir}")
        return tokenizer

print("✅ Tokenizer class defined successfully!")

✅ Tokenizer class defined successfully!


In [30]:
# Set paths
base_dir = os.getcwd()
processed_dir = os.path.join(base_dir, '..', 'data', 'processed')

# Load data
print("📂 Loading preprocessed data...")
train_df = pd.read_csv(os.path.join(processed_dir, 'train.csv'))
val_df = pd.read_csv(os.path.join(processed_dir, 'val.csv'))
test_df = pd.read_csv(os.path.join(processed_dir, 'test.csv'))

print(f"Train: {len(train_df)} samples")
print(f"Val: {len(val_df)} samples") 
print(f"Test: {len(test_df)} samples")

train_df.head()

📂 Loading preprocessed data...
Train: 8593 samples
Val: 1074 samples
Test: 1075 samples


,arabic,english
0,لقد قال انه سعيد,he said that he was happy
1,من الجيد معرفه ذلك,thats good to know
2,لست متاكدا حيال ما كان يتحدث توم عنه,im not sure what tom was talking about
3,زرنا طوكيو مرات عديده,we have been to tokyo many times
4,يصنع النحل العسل,bees make honey


In [32]:
# Initialize tokenizer
print("🔧 Initializing tokenizer...")
tokenizer = TranslationTokenizer(max_len=MAX_SEQ_LEN)

# Build vocabulary from training data only
print("\n📖 Building vocabulary from training data...")
tokenizer.build_vocabulary(
    arabic_sentences=train_df['arabic'].tolist(),
    english_sentences=train_df['english'].tolist(),
    min_freq=MIN_WORD_FREQ,
    max_size=MAX_VOCAB_SIZE
)

# Show vocab stats
print(f"\n📊 Vocabulary Statistics:")
print(f"   Arabic - Unique words: {len(tokenizer.ar_stoi)}")
print(f"   English - Unique words: {len(tokenizer.en_stoi)}")

🔧 Initializing tokenizer...

📖 Building vocabulary from training data...
✅ Arabic vocab size: 3550
✅ English vocab size: 2154

📊 Vocabulary Statistics:
   Arabic - Unique words: 3550
   English - Unique words: 2154


In [33]:
# Process all splits
print("🔄 Processing datasets...")

train_ar = tokenizer.process_dataframe(train_df, 'ar')
train_en = tokenizer.process_dataframe(train_df, 'en')

val_ar = tokenizer.process_dataframe(val_df, 'ar')
val_en = tokenizer.process_dataframe(val_df, 'en')

test_ar = tokenizer.process_dataframe(test_df, 'ar')
test_en = tokenizer.process_dataframe(test_df, 'en')

print(f"\n📐 Tensor shapes:")
print(f"   Train: {train_ar.shape}, {train_en.shape}")
print(f"   Val: {val_ar.shape}, {val_en.shape}")
print(f"   Test: {test_ar.shape}, {test_en.shape}")

🔄 Processing datasets...

📐 Tensor shapes:
   Train: torch.Size([8593, 50]), torch.Size([8593, 50])
   Val: torch.Size([1074, 50]), torch.Size([1074, 50])
   Test: torch.Size([1075, 50]), torch.Size([1075, 50])


In [34]:
# Save tensors
tensor_dir = os.path.join(base_dir, '..', 'data', 'tensors')
os.makedirs(tensor_dir, exist_ok=True)

torch.save(train_ar, os.path.join(tensor_dir, 'train_ar.pt'))
torch.save(train_en, os.path.join(tensor_dir, 'train_en.pt'))
torch.save(val_ar, os.path.join(tensor_dir, 'val_ar.pt'))
torch.save(val_en, os.path.join(tensor_dir, 'val_en.pt'))
torch.save(test_ar, os.path.join(tensor_dir, 'test_ar.pt'))
torch.save(test_en, os.path.join(tensor_dir, 'test_en.pt'))

# Save tokenizer
tokenizer_dir = os.path.join(base_dir, '..', 'models', 'tokenizer')
tokenizer.save(tokenizer_dir)

print("💾 All files saved successfully!")

✅ Tokenizer saved to c:\Users\LENOVO\Downloads\Transformer-NMT-Project-main(1)\Transformer-NMT-Project-main\Tokenization\..\models\tokenizer
💾 All files saved successfully!


In [35]:
# Test encoding/decoding
sample_idx = 0

original_ar = train_df.iloc[sample_idx]['arabic']
original_en = train_df.iloc[sample_idx]['english']

encoded_ar = train_ar[sample_idx].tolist()
encoded_en = train_en[sample_idx].tolist()

decoded_ar = tokenizer.decode(encoded_ar, 'ar')
decoded_en = tokenizer.decode(encoded_en, 'en')

print("🧪 Sample Test:")
print(f"\nOriginal Arabic:  {original_ar}")
print(f"Decoded Arabic:   {decoded_ar}")
print(f"\nOriginal English: {original_en}")
print(f"Decoded English:  {decoded_en}")

print("\n✅ Tokenizer working perfectly!")

🧪 Sample Test:

Original Arabic:  لقد قال انه سعيد
Decoded Arabic:   لقد قال انه سعيد

Original English: he said that he was happy
Decoded English:  he said that he was happy

✅ Tokenizer working perfectly!
